# Unit Tests — chromatic_matroids

Each section tests one module. Every cell prints verbose output and raises `AssertionError` on failure.
Run the entire notebook top-to-bottom; a clean run means all tests pass.

**Modules covered**
1. `Composition`
2. `SetComposition`
3. `Matroid` (construction, rank, independent sets, is_nested, extend, relabel)
4. Matroid generators (uniform, Schubert, nested, graphic)
5. `QSymFunction`
6. `NCQSymFunction`
7. `stable_matroids_setcompositions`
8. `chromatic_non_commutative_quasisymmetric_function` / `chromatic_quasisymmetric_function`
9. `compute_chromatic_polynomial`
10. Matrix construction utilities
11. `min_max_set_composition`

In [ ]:
from chromatic_matroids import (
    Composition,
    SetComposition,
    Matroid,
    QSymFunction,
    NCQSymFunction,
    uniform_matroid,
    schubert_matroid,
    nested_matroid,
    graphic_matroid,
    generate_schubert_matroids,
    generate_loopless_schubert_matroids,
    generate_loopless_nested_matroids,
    generate_nested_matroids_doublechains,
    stable_matroids_setcompositions,
    chromatic_quasisymmetric_function,
    chromatic_non_commutative_quasisymmetric_function,
    compute_chromatic_polynomial,
    from_set_to_set_composition,
    generate_valid_subsets,
    compute_lowerbound_matrix,
    compute_conjecture_matrix,
    min_max_set_composition,
    generate_min_max_set_compositions,
)

def ok(label):
    print(f"  PASS  {label}")

def section(title):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)

print("All imports successful.")

---
## 1. Composition

In [ ]:
section("Composition — construction")

# From list
c = Composition([2, 1, 3])
print(f"  Composition([2,1,3]) = {c}")
assert c.parts == [2, 1, 3]
assert c.n == 6
assert c.nparts == 3
ok("from list")

# From tuple
ct = Composition((1, 2))
assert ct.parts == [1, 2]
ok("from tuple")

# From string
cs = Composition("(2,1,3)")
print(f"  Composition('(2,1,3)') = {cs}")
assert cs.parts == [2, 1, 3]
ok("from string")

# Empty composition (composition of 0)
ce = Composition()
assert ce.parts == []
assert ce.n == 0
ok("empty composition")

# Empty composition from string
ces = Composition("()")
assert ces.parts == []
ok("empty composition from string")

# Equality
assert Composition([2, 1, 3]) == Composition([2, 1, 3])
assert not (Composition([2, 1, 3]) == Composition([3, 1, 2]))
ok("equality")

# Invalid inputs should raise
try:
    Composition([0, 1])   # 0 is not positive
    assert False, "Should have raised"
except ValueError:
    ok("ValueError on non-positive part")

try:
    Composition([1.5])    # float is not int
    assert False, "Should have raised"
except TypeError:
    ok("TypeError on float part")

In [ ]:
section("Composition — rest / prepend")

c = Composition([2, 1, 3])

r = c.rest()
print(f"  {c}.rest() = {r}")
assert r.parts == [1, 3]
assert r.n == 4
ok("rest")

p = c.prepend(4)
print(f"  {c}.prepend(4) = {p}")
assert p.parts == [4, 2, 1, 3]
assert p.n == 10
ok("prepend")

# Immutability: original must be unchanged
assert c.parts == [2, 1, 3]
ok("original unchanged after rest/prepend")

try:
    Composition().rest()
    assert False, "Should have raised on empty"
except Exception:
    ok("rest of empty raises")

In [ ]:
section("Composition — generate_all_composition")

# n=1 → [(1)]
c1 = Composition.generate_all_composition(1)
print(f"  n=1: {c1}")
assert len(c1) == 1
assert c1[0].parts == [1]
ok("n=1")

# n=2 → [(2), (1,1)]  (2 compositions)
c2 = Composition.generate_all_composition(2)
print(f"  n=2: {c2}")
assert len(c2) == 2
ok("n=2 count")

# n=3 → 4 compositions
c3 = Composition.generate_all_composition(3)
print(f"  n=3: {c3}")
assert len(c3) == 4
ok("n=3 count")

# n=4 → 8 compositions (2^{n-1})
c4 = Composition.generate_all_composition(4)
assert len(c4) == 8
ok("n=4 count = 8")

# Check each composition sums to n
for n in range(1, 6):
    for comp in Composition.generate_all_composition(n):
        assert comp.n == n, f"{comp} does not sum to {n}"
ok("all compositions sum to n (n=1..5)")

In [ ]:
section("Composition — quasi_shuffles")

# Shuffle of empty compositions
e = Composition()
c = Composition([1, 2])
qs = Composition.quasi_shuffles(e, c)
print(f"  qs((), (1,2)) = {qs}")
assert qs == {'(1,2)': 1}
ok("quasi_shuffle with left-empty")

qs2 = Composition.quasi_shuffles(c, e)
assert qs2 == {'(1,2)': 1}
ok("quasi_shuffle with right-empty")

# qs((1),(1)) should give {(2):1, (1,1):2}
c1 = Composition([1])
qs3 = Composition.quasi_shuffles(c1, c1)
print(f"  qs((1),(1)) = {qs3}")
assert qs3.get('(2)', 0) == 1, f"Expected (2):1, got {qs3}"
assert qs3.get('(1,1)', 0) == 2, f"Expected (1,1):2, got {qs3}"
ok("qs((1),(1))")

# Coefficients must be positive integers
for key, val in qs3.items():
    assert isinstance(val, int) and val > 0
ok("all quasi-shuffle coefficients are positive integers")

---
## 2. SetComposition

In [ ]:
section("SetComposition — construction")

# From list of lists
sc = SetComposition([[1, 4], [2], [3, 5, 6]])
print(f"  SetComposition([[1,4],[2],[3,5,6]]) = {sc}")
assert sc.parts == [[1, 4], [2], [3, 5, 6]]
assert set(sc.ground_set) == {1, 2, 3, 4, 5, 6}
ok("from list of lists")

# From string
sc_str = SetComposition("(1,4|2|3,5,6)")
print(f"  SetComposition('(1,4|2|3,5,6)') = {sc_str}")
assert sc_str.parts == [[1, 4], [2], [3, 5, 6]]
ok("from string")

# Empty
sc_e = SetComposition()
assert sc_e.parts == []
ok("empty")

# Non-disjoint parts should raise
try:
    SetComposition([[1, 2], [2, 3]])
    assert False, "Should have raised"
except ValueError:
    ok("ValueError on non-disjoint parts")

# String round-trip: str -> object -> str should be idempotent
sc2 = SetComposition([[3, 1], [2]])
sc3 = SetComposition(str(sc2))
print(f"  Round-trip: {sc2} -> '{str(sc2)}' -> {sc3}")
assert sc2.parts == sc3.parts
ok("string round-trip")

In [ ]:
section("SetComposition — equality")

sc_a = SetComposition([[1, 2], [3]])
sc_b = SetComposition([[1, 2], [3]])
sc_c = SetComposition([[1], [2, 3]])

print(f"  {sc_a} == {sc_b}: {sc_a == sc_b}")
print(f"  {sc_a} == {sc_c}: {sc_a == sc_c}")

assert sc_a == sc_b, "Same set compositions must be equal"
ok("equal set compositions")

assert not (sc_a == sc_c), "Different set compositions must not be equal"
ok("different set compositions are not equal")

In [ ]:
section("SetComposition — first / rest / prepend / alpha")

sc = SetComposition([[1, 4], [2], [3, 5, 6]])

first = sc.first()
print(f"  {sc}.first() = {first}")
assert first == frozenset({1, 4})
ok("first")

rest = sc.rest()
print(f"  {sc}.rest() = {rest}")
assert rest.parts == [[2], [3, 5, 6]]
ok("rest")

# Immutability: original unchanged
assert sc.parts == [[1, 4], [2], [3, 5, 6]]
ok("original unchanged after rest")

prepended = sc.prepend(frozenset({7, 8}))
print(f"  prepend({{7,8}}) -> {prepended}")
assert prepended.parts[0] == [7, 8]
ok("prepend")

alpha = sc.alpha()
print(f"  {sc}.alpha() = {alpha}")
assert alpha.parts == [2, 1, 3]  # sizes of [1,4], [2], [3,5,6]
ok("alpha")

try:
    SetComposition().rest()
    assert False
except Exception:
    ok("rest of empty raises")

In [ ]:
section("SetComposition — relabel")

sc = SetComposition([[1, 3], [2]])

# Dict relabeling
r1 = sc.relabel({1: 10, 2: 20, 3: 30})
print(f"  {sc}.relabel({{1:10,2:20,3:30}}) = {r1}")
assert set(r1.parts[0]) == {10, 30}
assert r1.parts[1] == [20]
ok("dict relabel")

# List relabeling (positional, by ground_set order)
r2 = sc.relabel([100, 200, 300])
print(f"  list relabel -> {r2}")
assert set(r2.ground_set) == {100, 200, 300}
ok("list relabel")

# None relabeling → standard {1, ..., n}
sc_odd = SetComposition([[3, 7], [5]])
r3 = sc_odd.relabel(None)
print(f"  {sc_odd}.relabel(None) = {r3}")
assert set(r3.ground_set) == {1, 2, 3}
ok("None relabel → {{1,...,n}}")

In [ ]:
section("SetComposition — generate_all_setcompositions")

# n=1 → 1 set composition: [[1]]
all_1 = SetComposition.generate_all_setcompositions(1)
print(f"  n=1: {all_1}")
assert len(all_1) == 1
ok("n=1 count")

# n=2 → 3: [[1,2]], [[2],[1]], [[1],[2]]
all_2 = SetComposition.generate_all_setcompositions(2)
print(f"  n=2: {all_2}")
assert len(all_2) == 3
ok("n=2 count")

# n=3 → 13 ordered set partitions
all_3 = SetComposition.generate_all_setcompositions(3)
assert len(all_3) == 13
ok("n=3 count = 13")

# n=4 → 75
all_4 = SetComposition.generate_all_setcompositions(4)
assert len(all_4) == 75
ok("n=4 count = 75")

# All generated set compositions must have ground set {1,...,n}
for sc in all_3:
    assert set(sc.ground_set) == {1, 2, 3}, f"Bad ground set: {sc.ground_set}"
ok("all ground sets are {{1,...,n}}")

# No two parts should share an element
for sc in all_4:
    seen = set()
    for part in sc.parts:
        for e in part:
            assert e not in seen, f"Duplicate element {e} in {sc}"
            seen.add(e)
ok("all parts are disjoint (n=4)")

In [ ]:
section("SetComposition — quasi_shuffles")

# Non-disjoint inputs must raise
sc1 = SetComposition([[1, 2], [3]])
sc2 = SetComposition([[2, 4], [5]])   # 2 appears in both
try:
    SetComposition.quasi_shuffles(sc1, sc2)
    assert False, "Should raise on non-disjoint"
except ValueError:
    ok("quasi_shuffle raises on non-disjoint")

# Shuffle of sc with empty
sc = SetComposition([[1], [2]])
se = SetComposition()
qs_left = SetComposition.quasi_shuffles(sc, se)
qs_right = SetComposition.quasi_shuffles(se, sc)
print(f"  qs({sc}, empty) = {qs_left}")
print(f"  qs(empty, {sc}) = {qs_right}")
assert str(sc) in qs_left
assert qs_left[str(sc)] == 1
assert str(sc) in qs_right
ok("quasi_shuffle with empty")

# Coefficients in a quasi-shuffle must be positive integers
sc_a = SetComposition([[1], [2]])
sc_b = SetComposition([[3]])
qs = SetComposition.quasi_shuffles(sc_a, sc_b)
print(f"  qs({sc_a}, {sc_b}) = {qs}")
total_weight = sum(qs.values())
print(f"  Total weight: {total_weight}")
for key, val in qs.items():
    assert isinstance(val, int) and val > 0, f"Bad coefficient {val} for {key}"
ok("all quasi-shuffle coefficients are positive integers")

---
## 3. Matroid

In [ ]:
section("Matroid — construction and validation")

# Valid matroid: U(3,2) — all pairs are bases
m = Matroid(
    frozenset([1, 2, 3]),
    {frozenset([1, 2]), frozenset([1, 3]), frozenset([2, 3])}
)
print(f"  U(3,2) ground_set={m.ground_set}, #bases={len(m.bases_sets)}")
assert m.ground_set == frozenset({1, 2, 3})
assert len(m.bases_sets) == 3
ok("U(3,2) constructs")

# Single-basis matroid (rank 0 on empty ground set)
m0 = Matroid(frozenset(), {frozenset()})
assert m0.ground_set == frozenset()
ok("rank-0 empty matroid")

# Exchange axiom violation should raise
try:
    Matroid(
        frozenset([1, 2, 3]),
        {frozenset([1, 2]), frozenset([3])}   # different sizes: not a valid matroid
    )
    assert False, "Should have raised"
except Exception:
    ok("exchange axiom violation raises")

# No bases → raises
try:
    Matroid(frozenset([1]), set())
    assert False
except Exception:
    ok("empty bases raises")

In [ ]:
section("Matroid — rank")

m = Matroid(
    frozenset([1, 2, 3]),
    {frozenset([1, 2]), frozenset([1, 3]), frozenset([2, 3])}
)

print(f"  rank(E) = {m.rank(m.ground_set)}")
assert m.rank(m.ground_set) == 2
ok("rank of ground set = 2")

print(f"  rank({{1,2}}) = {m.rank(frozenset([1,2]))}")
assert m.rank(frozenset([1, 2])) == 2
ok("rank of basis = 2")

print(f"  rank({{1}}) = {m.rank(frozenset([1]))}")
assert m.rank(frozenset([1])) == 1
ok("rank of singleton = 1")

print(f"  rank(empty) = {m.rank(frozenset())}")
assert m.rank(frozenset()) == 0
ok("rank of empty set = 0")

# Rank of the rank-0 matroid
m0 = Matroid(frozenset([1, 2]), {frozenset()})
assert m0.rank(frozenset([1, 2])) == 0
ok("rank of rank-0 matroid = 0")

In [ ]:
section("Matroid — independent_sets")

m = Matroid(
    frozenset([1, 2, 3]),
    {frozenset([1, 2]), frozenset([1, 3]), frozenset([2, 3])}
)
ind = m.independent_sets()
print(f"  Independent sets: {[sorted(s) for s in ind]}")

# Empty set is always independent
assert frozenset() in ind
ok("empty set is independent")

# All singletons are independent (rank >= 1)
for i in [1, 2, 3]:
    assert frozenset([i]) in ind
ok("all singletons are independent")

# All bases are independent
for basis in m.bases_sets:
    assert basis in ind
ok("all bases are independent")

# Ground set itself is NOT independent (rank 2, size 3)
assert frozenset([1, 2, 3]) not in ind
ok("ground set not independent")

# Total count for U(3,2): {} ∪ {singletons} ∪ {bases} = 1 + 3 + 3 = 7
assert len(ind) == 7
ok("total 7 independent sets")

In [ ]:
section("Matroid — is_nested")

# is_nested() checks the Bonin–de Mier flat-chain criterion:
# a matroid is nested iff its lattice of flats is totally ordered by inclusion.
#
# NOTE: This is DIFFERENT from the package's nested_matroid() parametric definition
# (chain of sets + rank constraints). The parametric definition is broader and includes
# matroids like U(3,2) that are NOT nested by the flat-chain criterion.

# U(3,2) is NOT nested: has three incomparable rank-1 flats {1}, {2}, {3}
m_u32 = Matroid(
    frozenset([1, 2, 3]),
    {frozenset([1, 2]), frozenset([1, 3]), frozenset([2, 3])}
)
result = m_u32.is_nested()
print(f"  U(3,2).is_nested() = {result}  (expected False)")
assert result == False
ok("U(3,2) is not nested")

# U(3,1) IS nested: only two flats (∅ and E), forming a trivial chain
# In U(n,1), adding any element to a non-empty set does not increase rank → no singleton flats
m_u31 = uniform_matroid(3, 1)
result2 = m_u31.is_nested()
print(f"  U(3,1).is_nested() = {result2}  (expected True)")
assert result2 == True
ok("U(3,1) is nested (only flats are ∅ and E)")

# U(2,0) IS nested: all elements are loops, only flat is E
m_u20 = uniform_matroid(2, 0)
print(f"  U(2,0).is_nested() = {m_u20.is_nested()}  (expected True)")
assert m_u20.is_nested() == True
ok("U(2,0) is nested (rank-0 matroid)")

# U(2,2) is NOT nested: has incomparable rank-1 flats {1} and {2}
m_u22 = uniform_matroid(2, 2)
print(f"  U(2,2).is_nested() = {m_u22.is_nested()}  (expected False)")
assert m_u22.is_nested() == False
ok("U(2,2) is not nested (free matroid on ≥2 elements has incomparable singleton flats)")

# sh({1,2,3},{2,3}) = U(3,2): not nested (same as m_u32 above)
m_sch = schubert_matroid(3, frozenset([2, 3]))
print(f"  sh(3,{{2,3}}).is_nested() = {m_sch.is_nested()}  (expected False)")
assert m_sch.is_nested() == False
ok("Schubert matroid sh(3,{2,3}) = U(3,2) is not nested")

# generate_loopless_nested_matroids uses the parametric flag definition (broader than B-dM).
# Of the 6 matroids generated for d=3, only U(3,1) satisfies the flat-chain criterion.
loopless_n3 = generate_loopless_nested_matroids(3)
bm_nested = [(m, m.is_nested()) for m in loopless_n3]
for m, nested in bm_nested:
    print(f"    bases={sorted([sorted(b) for b in m.bases_sets])}, is_nested={nested}")
bm_nested_count = sum(1 for _, n in bm_nested if n)
print(f"  B-dM nested among generate_loopless_nested_matroids(3): {bm_nested_count}/6")
assert bm_nested_count == 1, f"Expected 1, got {bm_nested_count}"
ok("1 of 6 parametric nested matroids (d=3) satisfies Bonin–de Mier flat-chain criterion")

In [ ]:
section("Matroid — extend (coloop)")

# Start with U(2,1) on {1,2}
m = uniform_matroid(2, 1)
print(f"  U(2,1) bases: {m.bases_sets}")

# Extend by element 3 as a coloop → rank goes from 1 to 2
m_ext = m.extend(3)
print(f"  After extend(3): ground_set={m_ext.ground_set}, bases={m_ext.bases_sets}")

assert m_ext.ground_set == frozenset({1, 2, 3})
assert m_ext.rank(m_ext.ground_set) == 2
# Every basis must contain element 3 (it's a coloop)
for basis in m_ext.bases_sets:
    assert 3 in basis, f"3 not in basis {basis}"
ok("extend adds a coloop in every basis")

# Original matroid unchanged
assert m.ground_set == frozenset({1, 2})
ok("original matroid unchanged after extend")

# Extending with an existing element should raise
try:
    m.extend(1)
    assert False
except Exception:
    ok("extend with existing element raises")

# The extended matroid is valid (matroid axioms hold)
# (Matroid constructor validates this automatically)
ok("extended matroid satisfies matroid axioms (validated in constructor)")

In [ ]:
section("Matroid — relabel")

m = Matroid(
    frozenset([1, 2, 3]),
    {frozenset([1, 2]), frozenset([1, 3]), frozenset([2, 3])}
)
bij = {1: 10, 2: 20, 3: 30}
m2 = m.relabel(bij)
print(f"  relabeled ground_set={m2.ground_set}, bases={m2.bases_sets}")

assert m2.ground_set == frozenset({10, 20, 30})
assert frozenset({10, 20}) in m2.bases_sets
assert frozenset({10, 30}) in m2.bases_sets
assert frozenset({20, 30}) in m2.bases_sets
ok("relabeled matroid has correct ground set and bases")

# Original unchanged
assert m.ground_set == frozenset({1, 2, 3})
ok("original matroid unchanged after relabel")

---
## 4. Matroid generators

In [ ]:
section("uniform_matroid")
from math import comb

for n, r in [(3, 0), (3, 1), (3, 2), (3, 3), (4, 2), (5, 3)]:
    m = uniform_matroid(n, r)
    print(f"  U({n},{r}): ground={len(m.ground_set)}, bases={len(m.bases_sets)}")
    assert m.ground_set == frozenset(range(1, n + 1))
    assert len(m.bases_sets) == comb(n, r), f"U({n},{r}): expected {comb(n,r)} bases"
    assert m.rank(m.ground_set) == r

ok("uniform_matroid: correct ground set, basis count, and rank")

# Bad inputs should raise
for bad_n, bad_r in [(-1, 0), (3, 4), (3, -1)]:
    try:
        uniform_matroid(bad_n, bad_r)
        assert False, f"Should raise for U({bad_n},{bad_r})"
    except Exception:
        pass
ok("uniform_matroid raises on invalid inputs")

In [ ]:
section("schubert_matroid")

# sh({1,...,4}, {2,3}): B={b1,b2} is a basis iff b1<=2, b2<=3
sch = schubert_matroid(4, frozenset([2, 3]))
expected_bases = [frozenset({1, 2}), frozenset({1, 3}), frozenset({2, 3})]
print(f"  sh(4,{{2,3}}) bases: {sorted([sorted(b) for b in sch.bases_sets])}")
assert len(sch.bases_sets) == 3
for eb in expected_bases:
    assert eb in sch.bases_sets, f"{eb} should be a basis"
ok("sh(4,{2,3}) has correct bases")

# Uniform matroid U(n,r) equals sh({1,...,n}, {n-r+1,...,n}) is nested
sch2 = schubert_matroid(3, frozenset([1, 2, 3]))
assert sch2.bases_sets == frozenset({frozenset({1, 2, 3})})
ok("sh(3,{1,2,3}) is the free matroid")

# generate_schubert_matroids(n) should give 2^n matroids
all_sch = generate_schubert_matroids(3)
print(f"  generate_schubert_matroids(3): {len(all_sch)} matroids")
assert len(all_sch) == 2**3  # one for each subset A of {1,...,n}
ok("generate_schubert_matroids(3) gives 2^3=8")

# generate_loopless_schubert_matroids(3) → loopless ones
loopless_sch = generate_loopless_schubert_matroids(3)
print(f"  generate_loopless_schubert_matroids(3): {len(loopless_sch)} matroids")
# Loopless means every element is in at least one basis
for m in loopless_sch:
    for e in m.ground_set:
        assert any(e in b for b in m.bases_sets), f"Element {e} is a loop in {m.bases_sets}"
ok("all loopless Schubert matroids are indeed loopless")

In [ ]:
section("nested_matroid")

# Simple nested matroid: chain [{1,2,3}], rank [2] → U(3,2)
m = nested_matroid(3, 2, (frozenset([1, 2, 3]),), (2,))
print(f"  ne(3,2,[{{1,2,3}}],[2]) bases: {sorted([sorted(b) for b in m.bases_sets])}")
assert m.bases_sets == {frozenset({1, 2}), frozenset({1, 3}), frozenset({2, 3})}
ok("single-level chain = uniform matroid")

# Two-level nested matroid
m2 = nested_matroid(4, 2, (frozenset([1, 2]), frozenset([1, 2, 3, 4])), (1, 2))
print(f"  ne(4,2,...) bases: {sorted([sorted(b) for b in m2.bases_sets])}")
# B must satisfy |B ∩ {1,2}| <= 1 and |B| = 2
for basis in m2.bases_sets:
    assert len(basis & frozenset([1, 2])) <= 1
ok("two-level nested matroid has correct rank conditions")

# generate_loopless_nested_matroids
loopless_n3 = generate_loopless_nested_matroids(3)
print(f"  generate_loopless_nested_matroids(3): {len(loopless_n3)} matroids")
assert len(loopless_n3) == 6  # known count
ok("generate_loopless_nested_matroids(3) gives 6")

# generate_nested_matroids_doublechains(0) edge case
dc0 = generate_nested_matroids_doublechains(0)
print(f"  generate_nested_matroids_doublechains(0): {dc0}")
assert len(dc0) == 1
ok("generate_nested_matroids_doublechains(0) returns 1 entry")

In [ ]:
section("graphic_matroid")

# Triangle K_3: 3 vertices, 3 edges, spanning trees have 2 edges
edges = [(1, 2), (1, 3), (2, 3)]
vertices = {1, 2, 3}
m = graphic_matroid(edges, vertices)
print(f"  K_3: ground_set={m.ground_set}, bases={m.bases_sets}")

# Rank should be |V| - components = 3 - 1 = 2
assert m.rank(m.ground_set) == 2
ok("K_3 graphic matroid has rank 2")

# All pairs of edges form spanning forests (no pair creates a cycle in a triangle... wait)
# Actually any 2 edges in K_3 form a spanning tree (they form a path on all 3 vertices)
assert len(m.bases_sets) == 3  # C(3,2) = 3 spanning trees
ok("K_3 has 3 spanning trees")

# Path graph P_3 (1-2-3): 2 edges, spanning tree = both edges, rank = 2
path_edges = [(1, 2), (2, 3)]
path_verts = {1, 2, 3}
m_path = graphic_matroid(path_edges, path_verts)
assert m_path.rank(m_path.ground_set) == 2
assert len(m_path.bases_sets) == 1  # only one spanning tree
ok("path P_3 has rank 2 and 1 spanning tree")

---
## 5. QSymFunction

In [ ]:
section("QSymFunction — construction")

# From Composition
c = Composition([2, 1])
q = QSymFunction(monomial_basis=c)
print(f"  QSym from Composition([2,1]): {q.coefficients}")
assert q.coefficients == {'(2,1)': 1}
ok("from Composition")

# From (Composition, int) tuple
q2 = QSymFunction(monomial_basis=(Composition([1, 3, 2]), -3))
assert q2.coefficients == {'(1,3,2)': -3}
ok("from (Composition, int) tuple")

# From dict
q3 = QSymFunction(monomial_basis={'(1,2)': 2, '(3,1)': -1})
assert q3.coefficients == {'(1,2)': 2, '(3,1)': -1}
ok("from dict")

# Zero: no argument
q_zero = QSymFunction()
assert q_zero.coefficients == {}
ok("zero QSymFunction")

# None
q_none = QSymFunction(monomial_basis=None)
assert q_none.coefficients == {}
ok("monomial_basis=None gives zero")

In [ ]:
section("QSymFunction — arithmetic (immutability check)")

q1 = QSymFunction(monomial_basis=Composition([1, 2]))
q2 = QSymFunction(monomial_basis=Composition([2, 1]))

# Addition
q_sum = q1 + q2
print(f"  M_(1,2) + M_(2,1) = {q_sum.coefficients}")
assert q_sum.coefficients == {'(1,2)': 1, '(2,1)': 1}
ok("addition")

# Immutability: q1 and q2 must be unchanged
assert q1.coefficients == {'(1,2)': 1}, f"q1 mutated: {q1.coefficients}"
assert q2.coefficients == {'(2,1)': 1}, f"q2 mutated: {q2.coefficients}"
ok("addition does not mutate operands")

# Adding same term accumulates coefficient
q_same = q1 + q1
assert q_same.coefficients == {'(1,2)': 2}
assert q1.coefficients == {'(1,2)': 1}  # unchanged
ok("adding same term accumulates coefficient, operand unchanged")

# Scalar multiplication
q_scaled = q1._scalarMultiple(5)
print(f"  5 * M_(1,2) = {q_scaled.coefficients}")
assert q_scaled.coefficients == {'(1,2)': 5}
assert q1.coefficients == {'(1,2)': 1}  # must not mutate q1
ok("scalar multiplication, operand unchanged")

# Multiplication (quasi-shuffle product)
qa = QSymFunction(monomial_basis=Composition([1]))
qb = QSymFunction(monomial_basis=Composition([1]))
qprod = qa * qb
print(f"  M_(1) * M_(1) = {qprod.coefficients}")
# qs((1),(1)) = {(2):1, (1,1):2}
assert qprod.coefficients.get('(2)', 0) == 1
assert qprod.coefficients.get('(1,1)', 0) == 2
ok("M_(1) * M_(1) quasi-shuffle product")

---
## 6. NCQSymFunction

In [ ]:
section("NCQSymFunction — construction")

# From SetComposition
sc = SetComposition([[1, 2], [3]])
nc = NCQSymFunction(monomial_basis=sc)
print(f"  NCQSym from SetComposition: {nc.coefficients}")
assert nc.coefficients == {'(1,2|3)': 1}
ok("from SetComposition")

# From string
nc2 = NCQSymFunction(monomial_basis="(1,2|3)")
assert nc2.coefficients == {'(1,2|3)': 1}
ok("from string")

# From dict
nc3 = NCQSymFunction(monomial_basis={'(1,2|3)': 2, '(1|2,3)': -1})
assert nc3.coefficients == {'(1,2|3)': 2, '(1|2,3)': -1}
ok("from dict")

# Zero
nc_zero = NCQSymFunction()
assert nc_zero.coefficients == {}
ok("zero NCQSymFunction")

In [ ]:
section("NCQSymFunction — arithmetic (immutability check)")

nc1 = NCQSymFunction(monomial_basis="(1|2)")
nc2 = NCQSymFunction(monomial_basis="(1,2)")

# Addition
nc_sum = nc1 + nc2
print(f"  M_(1|2) + M_(1,2) = {nc_sum.coefficients}")
assert nc_sum.coefficients == {'(1|2)': 1, '(1,2)': 1}
ok("addition")

# Critical: nc1 and nc2 must NOT be mutated
assert nc1.coefficients == {'(1|2)': 1}, f"nc1 mutated! Got {nc1.coefficients}"
assert nc2.coefficients == {'(1,2)': 1}, f"nc2 mutated! Got {nc2.coefficients}"
ok("addition does not mutate operands (mutation bug fixed)")

# Adding same term twice accumulates coefficient
nc_double = nc1 + nc1
assert nc_double.coefficients == {'(1|2)': 2}
assert nc1.coefficients == {'(1|2)': 1}  # still unchanged
ok("adding same NCQSym to itself, operand unchanged")

# Scalar multiplication
nc_scaled = nc1._scalarMultiple(5)
print(f"  5 * M_(1|2) = {nc_scaled.coefficients}")
assert nc_scaled.coefficients == {'(1|2)': 5}
assert nc1.coefficients == {'(1|2)': 1}  # must not mutate nc1
ok("scalar multiplication, operand unchanged (mutation bug fixed)")

# comu: collapses set composition to its underlying composition
# M_{(1,2|3)} maps to M_{(2,1)} since |{1,2}|=2, |{3}|=1
nc_sc = NCQSymFunction(monomial_basis=SetComposition([[1, 2], [3]]))
q_comm = nc_sc.comu()
print(f"  comu(M_{{1,2|3}}) = {q_comm.coefficients}")
assert q_comm.coefficients == {'(2,1)': 1}
ok("comu maps set-composition to underlying composition")

In [ ]:
section("NCQSymFunction — multiplication")

# M_{(1)} * M_{(2)}: quasi-shuffle of [(1)] and [(2)]
# The two set compositions must be disjoint, so we use {{1}} and {{2}}
nc_a = NCQSymFunction(monomial_basis=SetComposition([[1]]))
nc_b = NCQSymFunction(monomial_basis=SetComposition([[2]]))
nc_prod = nc_a * nc_b
print(f"  M_{{(1)}} * M_{{(2)}} = {nc_prod.coefficients}")

# qs([(1)], [(2)]) = {(1|2):1, (2|1):1, (1,2):1}
assert nc_prod.coefficients.get('(1|2)', 0) == 1
assert nc_prod.coefficients.get('(2|1)', 0) == 1
assert nc_prod.coefficients.get('(1,2)', 0) == 1
ok("NCQSym product via quasi-shuffle (undefined variable bug fixed)")

# Operands must be unchanged
assert nc_a.coefficients == {'(1)': 1}
assert nc_b.coefficients == {'(2)': 1}
ok("multiplication does not mutate operands")

---
## 7. stable_matroids_setcompositions

In [ ]:
section("stable_matroids_setcompositions")

# U(2,1): bases = {{1}, {2}}
m = uniform_matroid(2, 1)

# (2|1): weight vector [0,1]. Scores: {1}→1*0=0? No, wait:
# score(B) = sum over (index, part): index * |B ∩ part|
# For opi = (2|1): parts = [[2],[1]], indices 0 and 1
# score({1}) = 0*|{1}∩{2}| + 1*|{1}∩{1}| = 0 + 1 = 1
# score({2}) = 0*|{2}∩{2}| + 1*|{2}∩{1}| = 0 + 0 = 0
# Max score = 1, achieved by {1} only → stable
sc_21 = SetComposition([[2], [1]])
result = stable_matroids_setcompositions(m, sc_21)
print(f"  U(2,1), (2|1): stable={result}  (expected True)")
assert result == True
ok("U(2,1) stable for (2|1)")

# (1|2): same logic → stable (unique maximizer is {2})
sc_12 = SetComposition([[1], [2]])
result2 = stable_matroids_setcompositions(m, sc_12)
print(f"  U(2,1), (1|2): stable={result2}  (expected True)")
assert result2 == True
ok("U(2,1) stable for (1|2)")

# (1,2): single block, index 0 → all scores = 0, NOT unique maximizer
sc_12_single = SetComposition([[1, 2]])
result3 = stable_matroids_setcompositions(m, sc_12_single)
print(f"  U(2,1), (1,2): stable={result3}  (expected False)")
assert result3 == False
ok("U(2,1) not stable for (1,2)")

# Number of stable set compositions for U(2,1) equals
# the number of terms in its chromatic NCQSym function
nc = chromatic_non_commutative_quasisymmetric_function(m)
print(f"  chromatic NCQSym of U(2,1): {nc.coefficients}")
stable_count = sum(
    1 for sc in SetComposition.generate_all_setcompositions(2)
    if stable_matroids_setcompositions(m, sc)
)
assert stable_count == len(nc.coefficients)
ok("stable count matches NCQSym term count")

---
## 8. Chromatic quasisymmetric functions

In [ ]:
section("chromatic_non_commutative_quasisymmetric_function")

# U(2,1): by symmetry must have two terms, one for (2|1) and one for (1|2)
m = uniform_matroid(2, 1)
nc = chromatic_non_commutative_quasisymmetric_function(m)
print(f"  chromatic NCQSym of U(2,1): {nc.coefficients}")
assert set(nc.coefficients.keys()) == {'(2|1)', '(1|2)'}
assert nc.coefficients['(2|1)'] == 1
assert nc.coefficients['(1|2)'] == 1
ok("U(2,1) chromatic NCQSym = M_{(2|1)} + M_{(1|2)}")

section("chromatic_quasisymmetric_function (commutative)")

# comu of (M_{(2|1)} + M_{(1|2)}) = 2 * M_{(1,1)}
c = chromatic_quasisymmetric_function(m)
print(f"  chromatic QSym of U(2,1): {c.coefficients}")
assert c.coefficients == {'(1,1)': 2}
ok("U(2,1) chromatic QSym = 2 M_{(1,1)}")

# The dimension of the NCQSym span of loopless nested matroids of size d
# should equal d! (known result)
import numpy as np
from math import factorial

for d in [2, 3, 4]:
    matroids = generate_loopless_nested_matroids(d)
    nc_funcs = [chromatic_non_commutative_quasisymmetric_function(m) for m in matroids]
    all_keys = sorted(set(k for f in nc_funcs for k in f.coefficients))
    mat = [[f.coefficients.get(k, 0) for k in all_keys] for f in nc_funcs]
    rank = np.linalg.matrix_rank(mat)
    print(f"  d={d}: {len(matroids)} matroids, NCQSym matrix rank={rank}, expected={factorial(d)}")
    assert rank == factorial(d), f"d={d}: rank {rank} != {factorial(d)}"
ok("NCQSym rank = d! for d=2,3,4")

# The QSym span has dimension 2^{d-1}
for d in [2, 3, 4]:
    matroids = generate_loopless_nested_matroids(d)
    q_funcs = [chromatic_quasisymmetric_function(m) for m in matroids]
    all_keys = sorted(set(k for f in q_funcs for k in f.coefficients))
    mat = [[f.coefficients.get(k, 0) for k in all_keys] for f in q_funcs]
    rank = np.linalg.matrix_rank(mat)
    print(f"  d={d}: QSym matrix rank={rank}, expected={2**(d-1)}")
    assert rank == 2**(d-1), f"d={d}: rank {rank} != {2**(d-1)}"
ok("QSym rank = 2^{d-1} for d=2,3,4")

---
## 9. compute_chromatic_polynomial

In [ ]:
section("compute_chromatic_polynomial")

# U(2,1): χ(q) = q - 1 → coefficients [-1, 1]
p = compute_chromatic_polynomial(uniform_matroid(2, 1))
print(f"  χ(U(2,1), q) coefficients: {p}")
assert p == [-1, 1], f"Expected [-1, 1], got {p}"
ok("U(2,1): q - 1")

# U(3,2): χ(q) = q^2 - 3q + 2 → [2, -3, 1]
p2 = compute_chromatic_polynomial(uniform_matroid(3, 2))
print(f"  χ(U(3,2), q) coefficients: {p2}")
assert p2 == [2, -3, 1], f"Expected [2, -3, 1], got {p2}"
ok("U(3,2): q^2 - 3q + 2")

# U(3,0): rank-0 matroid, χ(q) = q^0 · (all-subset sum) ...
# Actually χ(U(n,0), q) = (1-1)^n · q^0 for n>0 = 0, but rank = 0 so exponent always 0
# Let's just check degree equals rank
for n, r in [(2, 0), (3, 1), (4, 2), (4, 4)]:
    m = uniform_matroid(n, r)
    poly = compute_chromatic_polynomial(m)
    print(f"  χ(U({n},{r}), q): {poly}")
    assert len(poly) == r + 1, f"Degree should be {r}, poly len = {len(poly)}"
ok("chromatic polynomial has degree = rank(M)")

# The leading coefficient is always 1
for n, r in [(2, 1), (3, 2), (4, 3)]:
    poly = compute_chromatic_polynomial(uniform_matroid(n, r))
    assert poly[-1] == 1, f"Leading coeff should be 1 for U({n},{r})"
ok("leading coefficient is always 1")

# Evaluate at q=1: χ(M, 1) = (-1)^r · β(M) where β is the beta invariant
# For a loop-free connected matroid, χ(M, 1) = 0
# Simple check: U(2,1) at q=1 → 1 - 1 = 0
p_u21 = compute_chromatic_polynomial(uniform_matroid(2, 1))
val_at_1 = sum(c * (1**k) for k, c in enumerate(p_u21))
assert val_at_1 == 0
ok("χ(U(2,1), 1) = 0")

---
## 10. Matrix construction utilities

In [ ]:
section("from_set_to_set_composition")

# from_set_to_set_composition({1,3}, 4) should produce a valid SetComposition on {1,2,3,4}
sc = from_set_to_set_composition({1, 3}, 4)
print(f"  from_set_to_set_composition({{1,3}}, 4) = {sc}")
assert set(sc.ground_set) == {1, 2, 3, 4}
ok("ground set is {1,...,d}")

sc2 = from_set_to_set_composition({1, 2, 3, 4}, 4)
print(f"  full set: {sc2}")
assert set(sc2.ground_set) == {1, 2, 3, 4}
ok("full set input")

section("generate_valid_subsets")

vs3 = generate_valid_subsets(3)
print(f"  valid subsets (d=3): {vs3}")
assert len(vs3) == 5  # known value
ok("generate_valid_subsets(3) gives 5")

# Full set is always included
assert (1, 2, 3) in vs3
ok("full set always in valid subsets")

section("compute_lowerbound_matrix")

import numpy as np

lb2 = compute_lowerbound_matrix(2)
print(f"  lowerbound matrix (d=2): {lb2}")
assert len(lb2) == len(lb2[0])  # must be square
rank2 = np.linalg.matrix_rank(lb2)
assert rank2 == len(lb2)  # must be full rank
ok("lowerbound_matrix(2) is square and full rank")

lb3 = compute_lowerbound_matrix(3)
rank3 = np.linalg.matrix_rank(lb3)
assert rank3 == len(lb3)
ok("lowerbound_matrix(3) is square and full rank")

section("compute_conjecture_matrix")

mat2, sc_list2, matroids2 = compute_conjecture_matrix(2)
print(f"  conjecture matrix (d=2): shape {len(mat2)}x{len(mat2[0]) if mat2 else 0}")
assert len(mat2) == len(mat2[0])  # must be square
rank_c2 = np.linalg.matrix_rank(mat2)
assert rank_c2 == len(mat2)
ok("conjecture_matrix(2) is square and full rank")

mat3, sc_list3, matroids3 = compute_conjecture_matrix(3)
rank_c3 = np.linalg.matrix_rank(mat3)
assert rank_c3 == len(mat3)
ok("conjecture_matrix(3) is square and full rank")

---
## 11. min_max_set_composition

In [ ]:
section("min_max_set_composition")

# The identity permutation (1,2,3) has no descents → one block
sc = min_max_set_composition((1, 2, 3))
print(f"  min_max((1,2,3)) = {sc}")
assert len(sc.parts) == 1
assert set(sc.ground_set) == {1, 2, 3}
ok("identity permutation → single block")

# (3,2,1): two descents at positions 0 and 1 → three blocks of size 1 each
sc2 = min_max_set_composition((3, 2, 1))
print(f"  min_max((3,2,1)) = {sc2}")
assert len(sc2.parts) == 3
ok("reverse permutation → three singleton blocks")

# (1,3,2): one descent at position 1 → two blocks [{1,3},{2}]
sc3 = min_max_set_composition((1, 3, 2))
print(f"  min_max((1,3,2)) = {sc3}")
assert len(sc3.parts) == 2
assert set(sc3.parts[0]) == {1, 3}
assert sc3.parts[1] == [2]
ok("(1,3,2) → [{1,3},{2}]")

# generate_min_max_set_compositions gives one composition per permutation
comps = generate_min_max_set_compositions(3)
from math import factorial
print(f"  generate_min_max_set_compositions(3): {len(comps)} compositions")
assert len(comps) == factorial(3)
ok("one min-max composition per permutation")

# All have ground set {1,...,n}
for sc in comps:
    assert set(sc.ground_set) == {1, 2, 3}
ok("all min-max compositions have ground set {1,...,n}")

---
## Summary

In [ ]:
print("\n" + "=" * 60)
print("  ALL TESTS PASSED")
print("=" * 60)